# 11 - Utilities Reference

> **Related**: This notebook demonstrates the use of sqlseed's internal utility classes, including MetricsCollector, sql_safe, Progress, Logger, and schema_helpers.

## What You Will Learn

- MetricsCollector performance metrics
- sql_safe SQL injection protection
- Progress multi-backend progress system
- Logger structlog logging
- schema_helpers AUTOINCREMENT detection

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| **→ 11** | **Utilities Reference** | **Utils** | **01** |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Internal Utilities | `src/sqlseed/_utils/` | `progress.py`, `sql_safe.py`, `metrics.py` |

## 1. MetricsCollector Performance Metrics

MetricsCollector collects and aggregates performance metrics, supporting filtering and statistics by name.

In [2]:
from sqlseed._utils.metrics import MetricsCollector

metrics = MetricsCollector()

metrics.record("fill_time", 1.234)
metrics.record("fill_time", 0.567)
metrics.record("fill_time", 2.891)
metrics.record("insert_count", 1000)
metrics.record("insert_count", 2000)

print("MetricsCollector API:")
print("  record('fill_time', 1.234) → records a metric")
print(f"  get_entries('fill_time') → {len(metrics.get_entries('fill_time'))} entries")
print(f"  get_entries() → {len(metrics.get_entries())} entries (all)")

summary = metrics.summary()
print("\n  summary():")
for name, stats in summary.items():
    print(f"    {name}: count={stats['count']}, avg={stats['avg']:.3f}, min={stats['min']:.3f}, max={stats['max']:.3f}")

metrics.clear()
print(f"\n  clear() → {len(metrics.get_entries())} entries (cleared)")

MetricsCollector API:
  record('fill_time', 1.234) → records a metric
  get_entries('fill_time') → 3 entries
  get_entries() → 5 entries (all)

  summary():
    fill_time: count=3, avg=1.564, min=0.567, max=2.891
    insert_count: count=2, avg=1500.000, min=1000.000, max=2000.000

  clear() → 0 entries (cleared)


## 2. sql_safe SQL Injection Protection

The sql_safe module provides SQL identifier escaping and validation to prevent SQL injection.

In [3]:
from sqlseed._utils.sql_safe import build_insert_sql, quote_identifier, validate_table_name

print("sql_safe — SQL injection protection:\n")

print(f"  quote_identifier('table_name') → {quote_identifier('table_name')}")
print("  quote_identifier('table\"name') → " + quote_identifier('table\"name'))

print(f"\n  validate_table_name('users') → {validate_table_name('users')}")

sql = build_insert_sql("users", ["name", "email", "age"])
print("\n  build_insert_sql('users', ['name', 'email', 'age']):")
print(f"    {sql}")

sql_safe — SQL injection protection:

  quote_identifier('table_name') → "table_name"
  quote_identifier('table"name') → "table""name"

  validate_table_name('users') → "users"

  build_insert_sql('users', ['name', 'email', 'age']):
    INSERT INTO "users" ("name", "email", "age") VALUES (?, ?, ?)


## 3. Progress Multi-Backend System

sqlseed uses the Strategy Pattern to implement cross-environment progress bars:
- **Terminal**: `RichProgressBackend` (based on Rich)
- **Jupyter**: `TqdmNotebookBackend` (based on tqdm.auto)
- **No UI**: `NullProgressBackend` (zero overhead)

`create_progress()` auto-detects the runtime environment and selects the appropriate backend.

In [4]:
from sqlseed._utils.progress import create_progress

# create_progress() returns a multi-backend Progress context manager
# Used internally by fill_table for batch progress display
progress = create_progress()
print(f'Progress type: {type(progress).__name__}')
print()
print('Usage in fill_table:')
print('  with create_progress() as progress:')
print('      task = progress.add_task("Filling...", total=count)')
print('      for batch in data_stream.generate():')
print('          progress.update(task, advance=len(batch))')
print()
print('Columns: Spinner, Bar, Percentage, Count, Speed, Time Remaining')

Progress type: TqdmNotebookBackend

Usage in fill_table:
  with create_progress() as progress:
      task = progress.add_task("Filling...", total=count)
      for batch in data_stream.generate():
          progress.update(task, advance=len(batch))

Columns: Spinner, Bar, Percentage, Count, Speed, Time Remaining


## 4. Logger structlog Logging

sqlseed uses structlog for structured logging; the log level is controlled via `GeneratorConfig.log_level`.

In [5]:
from sqlseed.config.models import GeneratorConfig

print("Logger — structlog logging:")
print("  sqlseed uses structlog for structured logging")
print("  Log level is controlled via GeneratorConfig.log_level")
print("  Default: INFO")

config = GeneratorConfig(db_path=str(db_path), log_level="DEBUG")
print(f"\n  GeneratorConfig(log_level='DEBUG') → {config.log_level}")

Logger — structlog logging:
  sqlseed uses structlog for structured logging
  Log level is controlled via GeneratorConfig.log_level
  Default: INFO

  GeneratorConfig(log_level='DEBUG') → DEBUG


## 5. schema_helpers AUTOINCREMENT Detection

schema_helpers provides AUTOINCREMENT primary key detection, used by ColumnMapper Level 1 to decide whether to skip generation.

In [6]:

print("schema_helpers — AUTOINCREMENT detection:")
print("  detect_autoincrement(execute_fn, table_name, column_name) → detects whether a column is an auto-increment primary key")
print("  Used by ColumnMapper Level 1 to decide whether to skip generation")

with sqlseed.connect(str(db_path)) as orch:
    col_info = orch.get_column_info("organizations")
    for col in col_info:
        if col.is_primary_key:
            print(f"\n  {col.name}: is_pk={col.is_primary_key}, is_autoincrement={col.is_autoincrement}")

schema_helpers — AUTOINCREMENT detection:
  detect_autoincrement(execute_fn, table_name, column_name) → detects whether a column is an auto-increment primary key
  Used by ColumnMapper Level 1 to decide whether to skip generation

  org_code: is_pk=True, is_autoincrement=False


## 6. ExpressionEngine Expression Engine

ExpressionEngine uses simpleeval to execute safe Python expressions, providing 21 whitelisted functions.

In [7]:
from sqlseed.core.expression import ExpressionEngine

engine = ExpressionEngine()

# Basic arithmetic
print(f"2 + 3 = {engine.evaluate('a + b', {'a': 2, 'b': 3})}")

# String operations
print(f"upper: {engine.evaluate('name.upper()', {'name': 'hello'})}")

# Safe functions (21 available)
print(f"abs(-5): {engine.evaluate('abs(x)', {'x': -5})}")
print(f"len: {engine.evaluate('len(s)', {'s': 'abc'})}")
print(f"min(1,2,3): {engine.evaluate('min(a,b,c)', {'a': 1, 'b': 2, 'c': 3})}")

2 + 3 = 5
upper: HELLO
abs(-5): 5
len: 3
min(1,2,3): 1


## ✅ Summary

| Tool | Function | Status |
|---|---|---|
| MetricsCollector | Performance metrics | ✅ |
| sql_safe | SQL injection protection | ✅ |
| Progress | Multi-backend progress system | ✅ |
| Logger | structlog logging | ✅ |
| schema_helpers | AUTOINCREMENT detection | ✅ |

**Next**: [12-testing-patterns.ipynb](12-testing-patterns.ipynb) — Testing Integration Patterns

In [8]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
